# ⏩ Quy Trình Công Việc Tuần Tự Với Microsoft Foundry (Python)

## 📋 Hướng dẫn Xử lý Tuần Tự Nâng Cao

Sổ tay này trình bày **mẫu quy trình công việc tuần tự** sử dụng Microsoft Agent Framework. Bạn sẽ học cách xây dựng các pipeline xử lý đa bước tinh vi, nơi các tác nhân thực thi theo thứ tự cụ thể, trao đổi dữ liệu và ngữ cảnh giữa các giai đoạn.

> **Ghi chú di chuyển:** Mẫu này trước đây tham chiếu đến GitHub Models. GitHub Models đã bị ngừng sử dụng (ngừng hoạt động vào tháng 7 năm 2026), nên hiện nay nó dùng **Microsoft Foundry** thông qua `FoundryChatClient`, nhắm tới Azure OpenAI **Responses API**.

## 🎯 Mục tiêu học tập

### 🔄 **Mẫu Xử lý Tuần tự**
- **Thiết kế Quy trình Tuyến tính**: Tạo các pipeline xử lý từng bước
- **Quản lý Dòng dữ liệu**: Chuyển thông tin giữa các tác nhân theo tuần tự
- **Xử lý theo Giai đoạn-Cổng**: Thực thi các điểm kiểm tra và giai đoạn xác thực
- **Theo dõi Tiến trình**: Giám sát thực thi quy trình và kết quả trung gian

### 🏗️ **Kiến trúc Pipeline Doanh nghiệp**
- **Mô hình Hóa Quy trình Kinh doanh**: Biểu diễn quy trình thực tế thành các workflows tác nhân
- **Đảm bảo Chất lượng**: Xác thực và xem xét nhiều giai đoạn
- **Xử lý Tài liệu**: Phân tích và chuyển đổi tài liệu tuần tự
- **Sản xuất Nội dung**: Quy trình biên tập với các bước xem xét và phê duyệt

### 📊 **Tính năng Quy trình Tiên tiến**
- **Bảo tồn Ngữ cảnh**: Duy trì trạng thái qua các giai đoạn workflow
- **Phát tán Lỗi**: Xử lý thất bại trong quá trình xử lý tuần tự
- **Tối ưu Hiệu năng**: Mẫu thực thi tuần tự hiệu quả
- **Dấu vết Kiểm toán**: Theo dõi toàn diện các thao tác tuần tự

## ⚙️ Yêu cầu & Thiết lập

### 📦 **Phụ thuộc**
```bash
pip install agent-framework -U
```

### 🔑 **Cấu hình**

Đăng nhập với Azure CLI (`az login`) để `AzureCliCredential` có thể xác thực, sau đó thiết lập chi tiết dự án Microsoft Foundry của bạn.

```env
AZURE_AI_PROJECT_ENDPOINT=https://<your-project>.services.ai.azure.com
AZURE_AI_MODEL_DEPLOYMENT_NAME=gpt-5-mini
```

## 🏢 **Trường hợp Sử dụng Quy trình Tuần tự Doanh nghiệp**

### 📝 **Pipeline Xử lý Tài liệu**
```
Raw Document → Content Extraction → Analysis → Validation → Final Output
```

### 🔍 **Quy trình Đảm bảo Chất lượng**
```
Initial Review → Technical Validation → Compliance Check → Final Approval
```

### 📰 **Pipeline Sản xuất Nội dung**
```
Research → Writing → Editing → Review → Publishing
```

### 💼 **Tự động hóa Quy trình Kinh doanh**
```
Data Collection → Processing → Analysis → Report Generation → Distribution
```

## 🎨 **Nguyên lý Thiết kế Quy trình Tuần tự**

- **🔗 Tiến trình Tuyến tính**: Mỗi giai đoạn phụ thuộc đầu ra của giai đoạn trước
- **📋 Quản lý Trạng thái**: Bảo tồn ngữ cảnh và dữ liệu qua tất cả các giai đoạn
- **🛡️ Xử lý Lỗi**: Quản lý thất bại một cách mềm dẻo ở bất kỳ giai đoạn nào
- **📊 Giám sát Tiến trình**: Theo dõi hoàn thành và hiệu năng ở mỗi giai đoạn
- **🔄 Tái sử dụng Giai đoạn**: Thiết kế các thành phần workflow có thể tái sử dụng

Hãy cùng xây dựng các quy trình xử lý tuần tự tinh vi! 🚀


In [ ]:
# Already covered by repo-level requirements.txt; left for reference.
# !pip install agent-framework -U

In [ ]:
from agent_framework import (
    WorkflowBuilder,
    WorkflowEvent,
    WorkflowViz,
)
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential


In [ ]:

import os
import base64
from dotenv import load_dotenv

In [ ]:
load_dotenv()

In [ ]:
# Configure the Microsoft Foundry client with keyless authentication.
# FoundryChatClient targets the Azure OpenAI Responses API.
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)


In [ ]:
SalesAgentName = "Sales-Agent"
SalesAgentInstructions = "You are my furniture sales consultant, you can find different furniture elements from the pictures and give me a purchase suggestion"

In [ ]:
PriceAgentName = "Price-Agent"
PriceAgentInstructions = """You are a furniture pricing specialist and budget consultant. Your responsibilities include:
        1. Analyze furniture items and provide realistic price ranges based on quality, brand, and market standards
        2. Break down pricing by individual furniture pieces
        3. Provide budget-friendly alternatives and premium options
        4. Consider different price tiers (budget, mid-range, premium)
        5. Include estimated total costs for room setups
        6. Suggest where to find the best deals and shopping recommendations
        7. Factor in additional costs like delivery, assembly, and accessories
        8. Provide seasonal pricing insights and best times to buy
        Always format your response with clear price breakdowns and explanations for the pricing rationale."""

In [ ]:
QuoteAgentName = "Quote-Agent"
QuoteAgentInstructions = """You are a assistant that create a quote for furniture purchase.
        1. Create a well-structured quote document that includes:
        2. A title page with the document title, date, and client name
        3. An introduction summarizing the purpose of the document
        4. A summary section with total estimated costs and recommendations
        5. Use clear headings, bullet points, and tables for easy readability
        6. All quotes are presented in markdown form"""

In [ ]:
sales_agent = provider.as_agent(
    name=SalesAgentName,
    instructions=SalesAgentInstructions,
)

price_agent = provider.as_agent(
    name=PriceAgentName,
    instructions=PriceAgentInstructions,
)

quote_agent = provider.as_agent(
    name=QuoteAgentName,
    instructions=QuoteAgentInstructions,
)


In [ ]:
workflow = (
    WorkflowBuilder(start_executor=sales_agent)
    .add_edge(sales_agent, price_agent)
    .add_edge(price_agent, quote_agent)
    .build()
)

In [ ]:
print("Generating workflow visualization...")
viz = WorkflowViz(workflow)
# Print out the mermaid string.
print("Mermaid string: \n=======")
print(viz.to_mermaid())
print("=======")
# Print out the DiGraph string.
print("DiGraph string: \n=======")
print(viz.to_digraph())
print("=======")
# SVG export needs the optional graphviz extra (`pip install graphviz`) plus the
# graphviz system binary; if it's not available, fall back to the text strings above.
try:
    svg_file = viz.export(format="svg")
    print(f"SVG file saved to: {svg_file}")
except ImportError as e:
    svg_file = None
    print(f"SVG export skipped (install graphviz to enable): {e}")

In [ ]:
class DatabaseEvent(WorkflowEvent): ...

In [ ]:
# Display the exported workflow SVG inline in the notebook

from IPython.display import SVG, display, HTML
import os

print(f"Attempting to display SVG file at: {svg_file}")

if svg_file and os.path.exists(svg_file):
    try:
        # Preferred: direct SVG rendering
        display(SVG(filename=svg_file))
    except Exception as e:
        print(f"⚠️ Direct SVG render failed: {e}. Falling back to raw HTML.")
        try:
            with open(svg_file, "r", encoding="utf-8") as f:
                svg_text = f.read()
            display(HTML(svg_text))
        except Exception as inner:
            print(f"❌ Fallback HTML render also failed: {inner}")
else:
    print("❌ SVG file not found. Ensure viz.export(format='svg') ran successfully.")

In [ ]:
image_path = "../imgs/home.png"
with open(image_path, "rb") as image_file:
    image_b64 = base64.b64encode(image_file.read()).decode()
image_uri = f"data:image/png;base64,{image_b64}"


In [ ]:
# Note: the original notebook used a multimodal message with an image of a
# living room. To keep the lesson focused on sequential workflow mechanics, this
# migration passes a textual description of the same scene as the workflow input.
# Agents accept a plain string, matching the basic and concurrent samples.
message = (
    "I am furnishing a modern living room and want pieces that fit a warm, "
    "inviting style: a comfortable three-seat sofa, two accent armchairs, a "
    "wooden coffee table, a TV stand, a floor lamp, and a soft area rug. "
    "Please find appropriate furniture and give the corresponding price for "
    "each piece, then produce a final purchase quote."
)

In [ ]:
# Workflow.run_stream is no longer part of the public API; the current Workflow
# returns a results object whose `get_outputs()` produces the AgentResponse from
# each output executor. The final stage (quote_agent) is the only output here.
events = await workflow.run(message)
outputs = events.get_outputs()
result = outputs[0].text if outputs else ""

In [ ]:
result.replace("None", "")

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Tuyên bố miễn trừ trách nhiệm**:
Tài liệu này đã được dịch bằng dịch vụ dịch thuật AI [Co-op Translator](https://github.com/Azure/co-op-translator). Mặc dù chúng tôi cố gắng đảm bảo độ chính xác, xin lưu ý rằng bản dịch tự động có thể chứa lỗi hoặc sai sót. Tài liệu gốc bằng ngôn ngữ gốc nên được coi là nguồn tin chính thức. Đối với thông tin quan trọng, nên sử dụng dịch vụ dịch thuật chuyên nghiệp bởi con người. Chúng tôi không chịu trách nhiệm về bất kỳ hiểu lầm hoặc giải thích sai nào phát sinh từ việc sử dụng bản dịch này.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
